In [1]:
from pydantic import BaseModel
from typing import Literal, List

class CliamState(BaseModel):
    user_id: str
    image_paths: str
    user_claim: str
    claim_object: Literal["car","laptop", "package"]
    evidence_stander_met: bool
    evidence_standard_met_reason: str
    risk_flags: List[str]
    issue_type: str
    object_part: str
    claim_status: Literal["supported","contradicted","not_enough_information"]
    supporting_image_ids: List[str]
    valid_image: bool
    severity: Literal["none","low","medium","high","unknown"]

In [2]:
# Dir paths
import os
BASE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
PARENT_DIR = os.path.dirname(BASE_DIR)
CLAIMS_DIR = os.path.join(PARENT_DIR, "data","claims")

#CSV files path
# D:\python\AI\pynb\data\claims\claims.csv
claims_csv = os.path.join(CLAIMS_DIR, "claims_test.csv")
# D:\python\AI\pynb\data\claims\evidence_requirements.csv
ev_req_csv = os.path.join(CLAIMS_DIR, "evidence_requirements.csv")
# D:\python\AI\pynb\data\claims\output.csv
output_csv = os.path.join(CLAIMS_DIR, "output.csv")
# D:\python\AI\pynb\data\claims\user_history.csv
user_history_csv = os.path.join(CLAIMS_DIR, "user_history.csv")

In [ ]:
#Read CSVs using pandas
import pandas as pd

claims_df = pd.read_csv(claims_csv)
evidence_requirements_df = pd.read_csv(ev_req_csv)
user_history_df = pd.read_csv(user_history_csv)
uh_row = user_history_df[user_history_df["user_id"] == "user_001"]
uh_row.to_json(orient="records")


'[{"user_id":"user_001","past_claim_count":2,"accept_claim":2,"manual_review_claim":0,"rejected_claim":0,"last_90_days_claim_count":1,"history_flags":"none","history_summary":"Low-risk user with prior accepted car damage claims"}]'

In [7]:
from google.genai import types
def cook_img_part(image_path:str):
    with open(image_path, "rb") as f:
        image_part = types.Part.from_bytes(
            data=f.read(),
            mime_type="image/jpeg"
        )
    return image_part

ev_req_json_data = uh_row.to_json(orient="records")

for row in claims_df[0:5].itertuples():
    user_id = row.user_id
    claim_object = row.claim_object
    claim = row.user_claim
    user_history = user_history_df[user_history_df["user_id"] == row.user_id].to_json(orient="records")
    image_paths = row.image_paths.split(";")
    image_abs_path = list(map(lambda x: os.path.join(CLAIMS_DIR, x.replace('/', '\\')), image_paths))
    image_parts = list(map(cook_img_part, image_abs_path))
    out = f"""
    user_id : {user_id},
    claim_object: {claim_object},
    claim: {claim},
    user_history: {user_history},
    image_paths: {image_paths},
    image_abs_path: {image_abs_path},
    image_parts: {image_parts}
    """
    print(out)


    user_id : user_001,
    claim_object: car,
    claim: Customer: Hi, I found new damage on my car after it was parked outside overnight. | Support: Sorry to hear that. Can you describe what changed? | Customer: The back of the car has a dent now. It was not there before. | Support: Did anything else break or is it mostly body damage? | Customer: Mostly the rear bumper area. I attached the photo I took this morning.,
    user_history: [{"user_id":"user_001","past_claim_count":2,"accept_claim":2,"manual_review_claim":0,"rejected_claim":0,"last_90_days_claim_count":1,"history_flags":"none","history_summary":"Low-risk user with prior accepted car damage claims"}],
    image_paths: ['images/sample/case_001/img_1.jpg'],
    image_abs_path: ['d:\\python\\AI\\pynb\\data\\claims\\images\\sample\\case_001\\img_1.jpg'],
    image_parts: [Part(
  inline_data=Blob(
    data=b"RIFF\xfc\xcd\x00\x00WEBPVP8 \xf0\xcd\x00\x00p\xda\x02\x9d\x01*X\x02\x90\x01>)\x12\x87B\xa1\xa1\x10RY\xc6|\x18\x02\x84\xb

In [14]:
import os
import json
from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()

GL_GEN_AI_API_KEY=os.environ.get('GL_GEN_AI_API_KEY')

BASE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
PARENT_DIR = os.path.dirname(BASE_DIR)
claims_path = os.path.join(PARENT_DIR, "data","claims")

def cook_img_part(image_path:str):
    with open(image_path, "rb") as f:
        image_part = types.Part.from_bytes(
            data=f.read(),
            mime_type="image/jpeg"
        )
    return image_part

In [16]:
out_list = []
for row in claims_df[0:3].itertuples():
    print(row.user_id)
    user_id = row.user_id
    claim_object = row.claim_object
    claim = row.user_claim
    user_history = user_history_df[user_history_df["user_id"] == row.user_id].to_json(orient="records")
    image_paths = row.image_paths.split(";")
    image_abs_path = list(map(lambda x: os.path.join(CLAIMS_DIR, x.replace('/', '\\')), image_paths))
    image_parts = list(map(cook_img_part, image_abs_path))

    client = genai.Client(api_key=GL_GEN_AI_API_KEY)

    prompt = f"""
        You are an automated insurance claim fraud detection and evidence verification system.

        Your objective is to determine whether the provided IMAGE EVIDENCE supports the CUSTOMER CLAIM and identify potential fraud or risk indicators.

        ====================================================
        INPUT DATA
        ====================================================

        USER_ID:
        {user_id}

        CLAIM_OBJECT:
        {claim_object}

        CUSTOMER_CONVERSATION:
        {claim}

        REQUIRED_EVIDENCE_STANDARD:
        {ev_req_json_data}

        USER_HISTORY:
        {user_history}

        IMAGES:
        Multiple images are provided separately as visual evidence.

        ====================================================
        ANALYSIS TASKS
        ====================================================

        Perform the following checks in strict order.

        ----------------------------------------------------
        1. EVIDENCE SUFFICIENCY CHECK
        ----------------------------------------------------

        Determine whether the provided image set contains sufficient evidence to evaluate the claim according to REQUIRED_EVIDENCE_STANDARD.

        Set:

        evidence_standard_met = true
            if sufficient evidence exists

        evidence_standard_met = false
            if critical evidence is missing

        Provide short explanation.

        ----------------------------------------------------
        2. IMAGE VALIDITY CHECK
        ----------------------------------------------------

        Determine whether the provided images are suitable for automated review.

        Set valid_image = false if ANY of the following occur:

        - blurry image
        - cropped image
        - low resolution
        - low lighting or glare
        - wrong object shown
        - claimed damage region not visible
        - object partially obstructed
        - manipulated or suspicious image

        Otherwise:

        valid_image = true

        ----------------------------------------------------
        3. SUPPORTING IMAGE IDENTIFICATION
        ----------------------------------------------------

        If multiple images are provided:

        Identify which image(s) provide strongest evidence supporting claim.

        Return:

        supporting_image_ids = list of image identifiers

        Examples:

        ["image_1"]
        ["image_2","image_4"]

        If none:

        []

        ----------------------------------------------------
        4. DAMAGE IDENTIFICATION
        ----------------------------------------------------

        Identify damaged object.

        Return:

        claim_object

        Examples:

        car
        laptop
        package

        Identify EXACTLY ONE damaged part.

        IMPORTANT RULES:

        - choose ONLY ONE object_part
        - never return multiple object parts

        If unknown:

        object_part = unknown

        ----------------------------------------------------
        5. DAMAGE TYPE CLASSIFICATION
        ----------------------------------------------------

        Identify EXACTLY ONE issue type visible in image.

        Choose exactly one issue_type.

        If no visible damage:

        issue_type = none

        If uncertain:

        issue_type = unknown

        ----------------------------------------------------
        6. DAMAGE SEVERITY
        ----------------------------------------------------

        Estimate visible severity.

        Allowed values:

        none
        low
        medium
        high
        unknown

        Guideline:

        none = no damage visible
        low = minor cosmetic damage
        medium = visible functional damage
        high = severe structural damage
        unknown = cannot determine

        ----------------------------------------------------
        7. CLAIM VERIFICATION
        ----------------------------------------------------

        Compare image evidence against CUSTOMER_CONVERSATION.

        Return one final decision.

        supported
            visible image evidence matches claim

        contradicted
            image evidence conflicts with claim

        not_enough_information
            insufficient evidence for conclusion

        Provide concise justification.

        IMPORTANT:

        Base decision ONLY on visible evidence.

        Do NOT assume hidden damage.

        ----------------------------------------------------
        8. RISK ANALYSIS
        ----------------------------------------------------

        Analyze:

        - user_history
        - image quality
        - claim consistency
        - suspicious visual indicators

        Assign zero or more risk flags.

        Examples:

        frequent prior claims
        repeated suspicious claims
        poor quality evidence
        possible manipulated image
        claim does not match image

        ====================================================
        ALLOWED ENUM VALUES
        ====================================================

        claim_status:

        supported
        contradicted
        not_enough_information

        issue_type:

        dent
        scratch
        crack
        glass_shatter
        broken_part
        missing_part
        torn_packaging
        crushed_packaging
        water_damage
        stain
        none
        unknown

        OBJECT PART RULES

        If claim_object = car

        front_bumper
        rear_bumper
        door
        hood
        windshield
        side_mirror
        headlight
        taillight
        fender
        quarter_panel
        body
        unknown

        If claim_object = laptop

        screen
        keyboard
        trackpad
        hinge
        lid
        corner
        port
        base
        body
        unknown

        If claim_object = package

        box
        package_corner
        package_side
        seal
        label
        contents
        item
        unknown

        risk_flags:

        none
        blurry_image
        cropped_or_obstructed
        low_light_or_glare
        wrong_angle
        wrong_object
        wrong_object_part
        damage_not_visible
        claim_mismatch
        possible_manipulation
        non_original_image
        text_instruction_present
        user_history_risk
        manual_review_required

        ====================================================
        STRICT RULES
        ====================================================

        DO NOT:

        - assume damage not visible
        - infer hidden damage
        - invent facts not present in image
        - return markdown
        - return explanation outside JSON
        - wrap output in ```json

        Return ONLY valid JSON.

        ====================================================
        RETURN EXACTLY THIS SCHEMA
        ====================================================

        {{
        "user_id": string,

        "image_paths": [string],

        "user_claim": string,

        "evidence_standard_met": boolean,

        "evidence_standard_met_reason": string,

        "valid_image": boolean,

        "severity": "none|low|medium|high|unknown",

        "issue_type": string,

        "object_part": string,

        "claim_object": string,

        "risk_flags": [string],

        "claim_status": "supported|contradicted|not_enough_information",

        "claim_status_justification": string,

        "supporting_image_ids": [string]
        }}
    """

    response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                prompt,
                image_parts
            ],
            config={
                "response_mime_type": "application/json"
            }
        )
    res_dict = json.loads(response.text)
    out_list.append(res_dict)



user_001


user_002
user_004


In [17]:
for i in out_list:
    print(i)

{'user_id': 'user_001', 'image_paths': ['image_1'], 'user_claim': 'The back of the car has a dent now. It was not there before. Mostly the rear bumper area.', 'evidence_standard_met': True, 'evidence_standard_met_reason': 'A clear image of the claimed damaged area (rear of the car) is provided.', 'valid_image': True, 'severity': 'high', 'issue_type': 'dent', 'object_part': 'body', 'claim_object': 'car', 'risk_flags': ['none'], 'claim_status': 'supported', 'claim_status_justification': "The image clearly shows severe denting and structural damage to the rear body and bumper area of the car, consistent with the customer's claim.", 'supporting_image_ids': ['image_1']}
{'user_id': 'user_002', 'image_paths': ['image_1', 'image_2'], 'user_claim': 'Front side par mark aa gaya hai, bumper ke upar. Light theek hai, front bumper par scratch hai.', 'evidence_standard_met': True, 'evidence_standard_met_reason': 'Image 1 shows visible damage on the claimed object part (front bumper), allowing for c

In [ ]:
# response = client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents=[
#         prompt,
#         types.Part.from_bytes(
#             data=image_bytes,
#             mime_type="image/jpeg" )
#     ],
#     config={
#         "response_mime_type": "application/json"
#     }
# )
# response

In [ ]:
import json
res_dict = json.loads(response.text)
res_dict

{'user_id': 'user_001',
 'image_paths': ['image_1'],
 'user_claim': 'The back of the car has a dent now. It was not there before. Mostly the rear bumper area.',
 'evidence_standard_met': True,
 'evidence_standard_met_reason': 'The image clearly shows the rear bumper and trunk area of the car from an angle suitable for assessing deformation and damage, meeting the standards for general object part and car body panel assessment.',
 'valid_image': True,
 'severity': 'high',
 'issue_type': 'broken_part',
 'object_part': 'rear_bumper',
 'claim_object': 'car',
 'risk_flags': ['none'],
 'claim_status': 'supported',
 'claim_status_justification': "The image provides clear visual evidence of extensive and severe damage, including broken components and significant deformation, to the rear bumper area and trunk of the car. This directly supports the customer's claim of a dent on the back of the car, specifically in the rear bumper area, after being parked overnight.",
 'supporting_image_ids': ['i

In [ ]:
#Eample for parsing

from langchain.messages import HumanMessage, SystemMessage
import json
def claim_parse_node(user_claim):
    prompt = f"""
    Extract reported_parts, issue_type, severity_hint from this conversation:
    {user_claim}
    Return JSON: {{ "reported_parts": [...], "issue_type": "...", "severity_hint": "..." }}
    """
    resp = llm.invoke([SystemMessage(content="Parse claim into JSON"), HumanMessage(content=prompt)])
    parsed = json.loads(resp.content)
    return {"parsed_claim": parsed}

In [ ]:
out

{'parsed_claim': {'reported_parts': ['rear bumper', 'back of the car'],
  'issue_type': 'dent',
  'severity_hint': 'minor'}}